# 03 Búsqueda vectorial con TF-IDF

## Objetivo

- Calcular e interpretar TF, DF, IDF y TF-IDF.
- Implementar TF-IDF mediante operaciones explícitas y verificarlo con `TfidfVectorizer`.
- Calcular similitud coseno mediante el producto punto.
- Implementar BM25 y comprender sus componentes.
- Construir y comparar rankings para una misma consulta.

### 1. Recuperar y revisar el corpus

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Construya una tabla con:

**Identificador de documento | Título | Número de términos únicos**

Presente:

- Número de documentos.
- Tamaño del vocabulario.
- Longitud mínima, media y máxima de los documentos.

Conserve el texto original para mostrar fragmentos de los resultados. Todas las técnicas deben utilizar **los mismos documentos y términos procesados**.

In [ ]:
import os
import math
import string
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.datasets import load_files

# Carga del corpus de Project Gutenberg (descargado en la práctica anterior)
corpus = load_files('txt-files', encoding='utf-8', shuffle=False)

textos_originales = corpus.data
N = len(textos_originales)
print(f"Se cargaron {N} documentos.")

# Identificador de cada documento: nombre del archivo sin la extensión (ej. libro_11)
ids_documentos = []
for ruta in corpus.filenames:
    nombre_archivo = os.path.basename(ruta)
    ids_documentos.append(nombre_archivo.replace('.txt', ''))

# Título: se busca la línea que empieza con "Title:" en la cabecera de Gutenberg
def obtener_titulo(texto):
    for linea in texto.split('\n'):
        if linea.startswith('Title:'):
            return linea.replace('Title:', '').strip()
    return 'Sin título'

titulos = []
for texto in textos_originales:
    titulos.append(obtener_titulo(texto))

# Stopwords en inglés
stopwords = {"the", "and", "to", "of", "a", "i", "in", "it", "that", "was", "he", "is", "for", "as", "on", "with"}

# Preprocesamiento
def preprocesar(texto):
    # 1. Minúsculas
    texto_limpio = texto.lower()
    # 2. Quitar puntuación
    texto_limpio = texto_limpio.translate(str.maketrans('', '', string.punctuation))
    # 3. Dividir en lista de palabras (tokenizar)
    palabras = texto_limpio.split()
    # 4. Filtrar stopwords y números
    palabras_filtradas = [p for p in palabras if p not in stopwords and p.isalpha()]
    return palabras_filtradas

# Procesamos todos los documentos
documentos_procesados = []
terminos_por_documento = []  # conjunto de términos únicos de cada documento
vocabulario_set = set()

for texto in textos_originales:
    palabras = preprocesar(texto)
    documentos_procesados.append(palabras)
    terminos_por_documento.append(set(palabras))
    vocabulario_set.update(palabras)

# Tabla: Identificador | Título | Número de términos únicos
tabla_corpus = pd.DataFrame({
    'Identificador': ids_documentos,
    'Título': titulos,
    'Términos únicos': [len(terminos) for terminos in terminos_por_documento]
})

# Longitud de cada documento = número de términos después del preprocesamiento
longitudes = [len(doc) for doc in documentos_procesados]

print(f"Número de documentos: {N}")
print(f"Tamaño del vocabulario: {len(vocabulario_set)}")
print(f"Longitud mínima: {min(longitudes)} términos")
print(f"Longitud media: {sum(longitudes) / N:.2f} términos")
print(f"Longitud máxima: {max(longitudes)} términos")

tabla_corpus

Se cargaron 100 documentos.
Número de documentos: 100
Tamaño del vocabulario: 125315
Longitud mínima: 2231 términos
Longitud media: 67085.77 términos
Longitud máxima: 720393 términos


,Identificador,Título,Términos únicos
0,libro_1,The Declaration of Independence of the United ...,1150
1,libro_10,The King James Version of the Bible,12850
2,libro_100,The Complete Works of William Shakespeare,25580
3,libro_101,The Hacker Crackdown: Law and Disorder on the ...,12313
4,libro_11,Alice's Adventures in Wonderland,2838
...,...,...,...
95,libro_95,The prisoner of Zenda,5620
96,libro_96,The Monster Men,5880
97,libro_97,Flatland: A Romance of Many Dimensions,4866
98,libro_98,A Tale of Two Cities,10654


### 2. Implementación de Funciones

Programe funciones para obtener:

1. TF de cada término en cada documento.
2. DF de cada término en **todo el corpus**.
3. IDF de cada término.
4. Los seis pesos TF-IDF correspondientes.

Para facilitar la comparación posterior con scikit-learn, utilizaremos:

$TF(t,d)=f(t,d)$

$
IDF(t)=\ln\left(\frac{N}{DF(t)}\right)+1
$

$
w_{t,d}=TF(t,d)\times IDF(t)
$

Aquí, $f(t,d)$ es el número de apariciones y $N$ el número de documentos.

**Nota:** Ahora utilizamos logaritmo natural y sumamos uno al IDF. Esta es la fórmula de scikit-learn cuando `smooth_idf=False`. [sklearn](https://sklearn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html?utm_source=chatgpt.com)

In [2]:
# 1. TF: número de veces que aparece el término en el documento -> TF(t,d) = f(t,d)
def calcular_tf(termino, documento):
    return documento.count(termino)

# 2. DF: número de documentos de TODO el corpus que contienen el término
def calcular_df(termino, terminos_por_documento):
    df = 0
    for terminos_doc in terminos_por_documento:
        if termino in terminos_doc:
            df += 1
    return df

# 3. IDF(t) = ln(N / DF(t)) + 1
def calcular_idf(df, N):
    return math.log(N / df) + 1

# 4. Peso TF-IDF: w(t,d) = TF(t,d) * IDF(t)
def calcular_tfidf(tf, idf):
    return tf * idf

# Ejemplo: seis pesos TF-IDF (2 términos x 3 documentos)
terminos_ejemplo = ['alice', 'whale']
docs_ejemplo = ['libro_10', 'libro_11', 'libro_15']  # Biblia, Alicia, Moby Dick

filas = []
for termino in terminos_ejemplo:
    df = calcular_df(termino, terminos_por_documento)
    idf = calcular_idf(df, N)
    for id_doc in docs_ejemplo:
        i = ids_documentos.index(id_doc)
        tf = calcular_tf(termino, documentos_procesados[i])
        filas.append({
            'Término': termino,
            'Documento': id_doc,
            'Título': titulos[i],
            'TF': tf,
            'DF': df,
            'IDF': round(idf, 4),
            'TF-IDF': round(calcular_tfidf(tf, idf), 4)
        })

pd.DataFrame(filas)

,Término,Documento,Título,TF,DF,IDF,TF-IDF
0,alice,libro_10,The King James Version of the Bible,0,18,2.7148,0.0000
1,alice,libro_11,Alice's Adventures in Wonderland,383,18,2.7148,1039.7678
2,alice,libro_15,"Moby-Dick; or, The Whale",0,18,2.7148,0.0000
3,whale,libro_10,The King James Version of the Bible,2,11,3.2073,6.4145
4,whale,libro_11,Alice's Adventures in Wonderland,0,11,3.2073,0.0000
5,whale,libro_15,"Moby-Dick; or, The Whale",930,11,3.2073,2982.7657


### 3. Implementar TF-IDF para el corpus

1. Construir un vocabulario con un orden definido.
2. Contar las frecuencias TF.
3. Calcular DF contando cada documento una sola vez por término.
4. Calcular IDF con la función de la sección anterior.
5. Obtener la matriz documento-término TF-IDF.

In [3]:
# Funciones de la práctica anterior adaptadas a la nueva fórmula de IDF

def obtener_matriz_tf(vocabulario, documentos):
    matriz_tf = []
    for doc in documentos:
        conteo = Counter(doc)
        fila_tf = []
        for termino in vocabulario:
            frecuencia = conteo[termino]
            fila_tf.append(frecuencia)
        matriz_tf.append(fila_tf)
    return matriz_tf

def obtener_df(vocabulario, matriz_tf):
    # Cada documento se cuenta una sola vez por término (si TF > 0)
    lista_df = []
    num_documentos = len(matriz_tf)
    for j in range(len(vocabulario)):
        contador_docs = 0
        for i in range(num_documentos):
            if matriz_tf[i][j] > 0:
                contador_docs += 1
        lista_df.append(contador_docs)
    return lista_df

def obtener_idf(lista_df, N):
    # Se usa la función calcular_idf de la sección anterior
    lista_idf = []
    for df in lista_df:
        lista_idf.append(calcular_idf(df, N))
    return lista_idf

def calcular_matriz_tfidf(matriz_tf, lista_idf):
    matriz_tfidf = []
    for fila_tf in matriz_tf:
        fila_tfidf = []
        for j in range(len(lista_idf)):
            fila_tfidf.append(calcular_tfidf(fila_tf[j], lista_idf[j]))
        matriz_tfidf.append(fila_tfidf)
    return matriz_tfidf

# 1. Vocabulario con un orden definido (alfabético)
vocabulario = sorted(vocabulario_set)
print(f"Vocabulario: {len(vocabulario)} términos")

# 2. Frecuencias TF
print("Calculando matriz TF...")
matriz_tf = obtener_matriz_tf(vocabulario, documentos_procesados)

# 3. DF
print("Calculando DF...")
lista_df = obtener_df(vocabulario, matriz_tf)

# 4. IDF
print("Calculando IDF...")
lista_idf = obtener_idf(lista_df, N)

# 5. Matriz documento-término TF-IDF
print("Calculando matriz TF-IDF...")
matriz_tfidf = calcular_matriz_tfidf(matriz_tf, lista_idf)

# La pasamos a numpy para poder compararla después con scikit-learn
matriz_tfidf = np.array(matriz_tfidf)
print(f"Dimensiones de la matriz TF-IDF: {matriz_tfidf.shape}")

# Vista de una parte de la matriz (términos de ejemplo)
pd.DataFrame(matriz_tfidf, index=ids_documentos, columns=vocabulario)[['alice', 'whale', 'king', 'god', 'internet']].head(15)

Vocabulario: 125315 términos
Calculando matriz TF...
Calculando DF...
Calculando IDF...
Calculando matriz TF-IDF...
Dimensiones de la matriz TF-IDF: (100, 125315)


,alice,whale,king,god,internet
libro_1,0.000000,0.000000,1.478036,1.385662,0.000000
libro_10,0.000000,6.414550,3346.273053,6156.498402,0.000000
libro_100,84.158751,25.658199,4440.019546,1079.431073,0.000000
libro_101,0.000000,0.000000,28.082680,5.542650,118.570014
libro_11,1039.767798,0.000000,88.682148,0.000000,0.000000
libro_12,1194.511308,0.000000,91.638220,0.000000,0.000000
libro_13,0.000000,0.000000,1.478036,0.000000,0.000000
libro_14,2.714798,0.000000,25.126609,11.085300,0.000000
libro_15,0.000000,2982.765669,63.555539,153.808535,0.000000
libro_16,0.000000,0.000000,0.000000,0.000000,0.000000


### 4. Verificar con `TfidfVectorizer`

Construya una segunda representación utilizando la librería `TfidfVectorizer`, con esta configuración:

| Parámetro | Valor |
|---|---|
| `use_idf` | `True` |
| `smooth_idf` | `False` |
| `sublinear_tf` | `False` |
| `norm` | `None` |

Esta configuración permite comparar pesos sin normalización y con TF como conteo directo. [scikit-learn 1.9.1 documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html?highlight=vectorizer\&utm_source=chatgpt.com)

Aseguren que el vectorizador recibe exactamente los mismos tokens que su implementación.

Calcule la **máxima diferencia absoluta** entre ambas matrices.

Si no coinciden, investiguen el preprocesamiento, el orden de columnas, la fórmula de IDF y la normalización.

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Para que el vectorizador reciba exactamente los mismos tokens,
# le pasamos las listas ya preprocesadas y le decimos que no haga nada más
def sin_cambios(tokens):
    return tokens

vectorizador = TfidfVectorizer(
    analyzer=sin_cambios,  # usa nuestros tokens tal cual
    use_idf=True,
    smooth_idf=False,
    sublinear_tf=False,
    norm=None
)

matriz_sklearn = vectorizador.fit_transform(documentos_procesados)
print(f"Dimensiones de la matriz de sklearn: {matriz_sklearn.shape}")

Dimensiones de la matriz de sklearn: (100, 125315)


In [5]:
# Revisamos que el orden de las columnas (vocabulario) sea el mismo
vocabulario_sklearn = list(vectorizador.get_feature_names_out())
print("¿Mismo vocabulario y mismo orden?:", vocabulario_sklearn == vocabulario)

# Revisamos que los IDF sean iguales
print("Máxima diferencia en IDF:", np.max(np.abs(np.array(lista_idf) - vectorizador.idf_)))

# Máxima diferencia absoluta entre ambas matrices TF-IDF
diferencia_maxima = np.max(np.abs(matriz_tfidf - matriz_sklearn.toarray()))
print("Máxima diferencia absoluta entre matrices:", diferencia_maxima)

if diferencia_maxima < 1e-9:
    print("Las dos matrices coinciden.")
else:
    print("Las matrices NO coinciden, revisar preprocesamiento, orden de columnas, IDF o normalización.")

¿Mismo vocabulario y mismo orden?: True
Máxima diferencia en IDF: 0.0
Máxima diferencia absoluta entre matrices: 0.0
Las dos matrices coinciden.


### 5. Representar consultas y calcular similitud coseno

Defina **cuatro consultas** que representen necesidades de información diferentes.

Para cada consulta:

1. Escriban qué información espera encontrar el usuario.
2. Apliquen el mismo preprocesamiento de los documentos.
3. Construyan su vector TF-IDF utilizando **el vocabulario y los IDF del corpus**.
4. No recalculen IDF incorporando la consulta como otro documento.

Utilice `transform` del vectorizador ajustado.

**Similitud coseno**

La similitud coseno compara la orientación de los vectores de consulta y documento:

$
\operatorname{sim}(q,d)
=\frac{q\cdot d}{\|q\|\|d\|}
=\frac{\sum_i q_i d_i}
{\sqrt{\sum_i q_i^2}\sqrt{\sum_i d_i^2}}
$

Implemente este cálculo mediante el producto punto y las magnitudes de los vectores. Luego, comparen los resultados con `sklearn.metrics.pairwise.cosine_similarity`, que calcula ese producto punto normalizado. [sklearn](https://sklearn.org/stable/modules/metrics.html?utm_source=chatgpt.com)

Para una combinación consulta-documento, muestren:

**Términos compartidos | Productos parciales | Producto punto | Normas | Similitud**

In [6]:
# Cuatro consultas que representan necesidades de información diferentes
consultas = [
    {
        'consulta': 'whale hunting ship captain harpoon sea',
        'necesidad': 'Novelas de aventuras sobre la caza de ballenas en altamar.'
    },
    {
        'consulta': 'internet network email computer users',
        'necesidad': 'Guías o textos que expliquen qué es Internet y cómo usar el correo electrónico.'
    },
    {
        'consulta': 'tarzan jungle apes',
        'necesidad': 'Historias de Tarzán y sus aventuras en la selva.'
    },
    {
        'consulta': 'constitution congress president rights government',
        'necesidad': 'Documentos históricos sobre la Constitución y el gobierno de Estados Unidos.'
    }
]

# Aplicamos a cada consulta el MISMO preprocesamiento de los documentos
filas = []
for c in consultas:
    c['tokens'] = preprocesar(c['consulta'])
    tokens_en_vocabulario = [t for t in c['tokens'] if t in vocabulario_set]
    filas.append({
        'Consulta': c['consulta'],
        'Necesidad de información': c['necesidad'],
        'Tokens procesados': c['tokens'],
        'Tokens en el vocabulario': tokens_en_vocabulario
    })

pd.set_option('display.max_colwidth', None)
pd.DataFrame(filas)

,Consulta,Necesidad de información,Tokens procesados,Tokens en el vocabulario
0,whale hunting ship captain harpoon sea,Novelas de aventuras sobre la caza de ballenas en altamar.,"[whale, hunting, ship, captain, harpoon, sea]","[whale, hunting, ship, captain, harpoon, sea]"
1,internet network email computer users,Guías o textos que expliquen qué es Internet y cómo usar el correo electrónico.,"[internet, network, email, computer, users]","[internet, network, email, computer, users]"
2,tarzan jungle apes,Historias de Tarzán y sus aventuras en la selva.,"[tarzan, jungle, apes]","[tarzan, jungle, apes]"
3,constitution congress president rights government,Documentos históricos sobre la Constitución y el gobierno de Estados Unidos.,"[constitution, congress, president, rights, government]","[constitution, congress, president, rights, government]"


In [ ]:
# Diccionario para encontrar rápido la columna de cada término en el vocabulario
indice_termino = {}
for j, termino in enumerate(vocabulario):
    indice_termino[termino] = j

# Vector TF-IDF de una consulta usando el vocabulario y los IDF del CORPUS.
# No se recalcula el IDF: la consulta no se agrega como otro documento.
def vector_consulta(tokens):
    vector = np.zeros(len(vocabulario))
    conteo = Counter(tokens)
    for termino in conteo:
        if termino in indice_termino:  # los términos que no están en el vocabulario se ignoran
            j = indice_termino[termino]
            vector[j] = calcular_tfidf(conteo[termino], lista_idf[j])
    return vector

for c in consultas:
    # Versión explícita
    c['vector_manual'] = vector_consulta(c['tokens'])
    # Versión con la librería: transform (NO fit) del vectorizador ya ajustado con el corpus
    c['vector_sklearn'] = vectorizador.transform([c['tokens']])

    diferencia = np.max(np.abs(c['vector_manual'] - c['vector_sklearn'].toarray()[0]))
    print(f"Consulta '{c['consulta']}' -> diferencia máxima entre vectores: {diferencia}")

Consulta 'whale hunting ship captain harpoon sea' -> diferencia máxima entre vectores: 0.0
Consulta 'internet network email computer users' -> diferencia máxima entre vectores: 0.0
Consulta 'tarzan jungle apes' -> diferencia máxima entre vectores: 0.0
Consulta 'constitution congress president rights government' -> diferencia máxima entre vectores: 0.0


In [8]:
from sklearn.metrics.pairwise import cosine_similarity

TOLERANCIA = 1e-9

# Similitud coseno = producto punto / (norma de q * norma de d)
def similitud_coseno(q, d):
    producto_punto = np.dot(q, d)
    norma_q = math.sqrt(np.sum(q ** 2))
    norma_d = math.sqrt(np.sum(d ** 2))
    if norma_q == 0 or norma_d == 0:
        return 0.0
    return producto_punto / (norma_q * norma_d)

for c in consultas:
    # Versión explícita: comparamos la consulta con cada documento
    similitudes = []
    for i in range(N):
        similitudes.append(similitud_coseno(c['vector_manual'], matriz_tfidf[i]))
    c['sim_manual'] = np.array(similitudes)

    # Versión con la librería
    c['sim_sklearn'] = cosine_similarity(c['vector_sklearn'], matriz_sklearn)[0]

    diferencia = np.max(np.abs(c['sim_manual'] - c['sim_sklearn']))
    print(f"Consulta '{c['consulta']}' -> diferencia máxima de similitud: {diferencia:.2e}",
          "(coinciden)" if diferencia < TOLERANCIA else "(NO coinciden)")

Consulta 'whale hunting ship captain harpoon sea' -> diferencia máxima de similitud: 2.75e-14 (coinciden)
Consulta 'internet network email computer users' -> diferencia máxima de similitud: 1.55e-14 (coinciden)
Consulta 'tarzan jungle apes' -> diferencia máxima de similitud: 3.05e-15 (coinciden)
Consulta 'constitution congress president rights government' -> diferencia máxima de similitud: 1.18e-14 (coinciden)


In [9]:
# Detalle del cálculo para una combinación consulta-documento:
# la primera consulta y el documento más similar a ella
c = consultas[0]
i = int(np.argmax(c['sim_manual']))
q = c['vector_manual']
d = matriz_tfidf[i]

print(f"Consulta: '{c['consulta']}'")
print(f"Documento: {ids_documentos[i]} - {titulos[i]}\n")

# Términos compartidos y productos parciales q_i * d_i
terminos_compartidos = []
productos_parciales = []
filas = []
for termino in sorted(set(c['tokens'])):
    if termino in indice_termino:
        j = indice_termino[termino]
        if q[j] > 0 and d[j] > 0:
            terminos_compartidos.append(termino)
            productos_parciales.append(q[j] * d[j])
            filas.append({
                'Término': termino,
                'Peso en consulta (q_i)': round(q[j], 4),
                'Peso en documento (d_i)': round(d[j], 4),
                'Producto parcial (q_i * d_i)': round(q[j] * d[j], 4)
            })

display(pd.DataFrame(filas))

# Producto punto, normas y similitud
producto_punto = sum(productos_parciales)
norma_q = math.sqrt(np.sum(q ** 2))
norma_d = math.sqrt(np.sum(d ** 2))
similitud = producto_punto / (norma_q * norma_d)

resumen = pd.DataFrame([{
    'Términos compartidos': ', '.join(terminos_compartidos),
    'Productos parciales': [round(p, 2) for p in productos_parciales],
    'Producto punto': round(producto_punto, 4),
    'Normas (|q|, |d|)': (round(norma_q, 4), round(norma_d, 4)),
    'Similitud': round(similitud, 6),
    'Similitud sklearn': round(c['sim_sklearn'][i], 6)
}])
resumen

Consulta: 'whale hunting ship captain harpoon sea'
Documento: libro_15 - Moby-Dick; or, The Whale



,Término,Peso en consulta (q_i),Peso en documento (d_i),Producto parcial (q_i * d_i)
0,captain,1.7550,493.1613,865.5093
1,harpoon,4.5066,306.4459,1381.0164
2,hunting,1.6539,13.2314,21.8838
3,sea,1.3711,501.8093,688.0125
4,ship,1.6733,620.8108,1038.8304
5,whale,3.2073,2982.7657,9566.5495


,Términos compartidos,Productos parciales,Producto punto,"Normas (|q|, |d|)",Similitud,Similitud sklearn
0,"captain, harpoon, hunting, sea, ship, whale","[865.51, 1381.02, 21.88, 688.01, 1038.83, 9566.55]",13561.8019,"(6.4102, 8520.3683)",0.248305,0.248305


### 6. Construir el ranking usando TF-IDF

Para cada consulta, ordenen los documentos de mayor a menor similitud coseno y presenten el **top-5**:

**Posición | Identificador | Título | Puntuación | Fragmento**

Las versiones explícita y basada en bibliotecas deben producir las mismas puntuaciones dentro de la tolerancia establecida.

In [10]:
import re

# Fragmento del texto original donde aparece algún término de la consulta
def obtener_fragmento(texto, tokens, largo=200):
    # Saltamos la cabecera de Gutenberg (licencia) y la línea "*** START OF ... ***"
    inicio_libro = texto.find('*** START OF')
    if inicio_libro == -1:
        inicio_libro = 0
    else:
        inicio_libro = texto.find('\n', inicio_libro)  # empezamos en la línea siguiente
    cuerpo = texto[inicio_libro:]
    cuerpo_minusculas = cuerpo.lower()

    # Buscamos la primera aparición de un término de la consulta (palabra completa)
    for termino in tokens:
        encontrado = re.search(r'\b' + termino + r'\b', cuerpo_minusculas)
        if encontrado:
            inicio = max(0, encontrado.start() - 60)
            fragmento = cuerpo[inicio:inicio + largo]
            return '...' + ' '.join(fragmento.split()) + '...'

    # Si no se encuentra ningún término, mostramos el inicio del libro
    return ' '.join(cuerpo[:largo].split()) + '...'

for numero, c in enumerate(consultas, start=1):
    # Ordenamos los documentos de mayor a menor similitud y tomamos los 5 primeros
    orden = np.argsort(c['sim_manual'])[::-1]
    top5 = orden[:5]

    filas = []
    for posicion, i in enumerate(top5, start=1):
        filas.append({
            'Posición': posicion,
            'Identificador': ids_documentos[i],
            'Título': titulos[i],
            'Puntuación': round(c['sim_manual'][i], 6),
            'Puntuación sklearn': round(c['sim_sklearn'][i], 6),
            'Fragmento': obtener_fragmento(textos_originales[i], c['tokens'])
        })

    # Comprobamos que la versión explícita y la de sklearn dan las mismas puntuaciones
    diferencia = np.max(np.abs(c['sim_manual'] - c['sim_sklearn']))

    print(f"\nConsulta {numero}: '{c['consulta']}'")
    print(f"Necesidad: {c['necesidad']}")
    print(f"¿Puntuaciones iguales a sklearn (tolerancia {TOLERANCIA})?: {diferencia < TOLERANCIA}")
    display(pd.DataFrame(filas))


Consulta 1: 'whale hunting ship captain harpoon sea'
Necesidad: Novelas de aventuras sobre la caza de ballenas en altamar.
¿Puntuaciones iguales a sklearn (tolerancia 1e-09)?: True


,Posición,Identificador,Título,Puntuación,Puntuación sklearn,Fragmento
0,1,libro_15,"Moby-Dick; or, The Whale",0.248305,0.248305,"...OR, THE WHALE. BY HERMAN MELVILLE, AUTHO..."
1,2,libro_13,The Hunting of the Snark: An Agony in Eight Fits,0.024701,0.024701,...THE HUNTING OF THE SNARK an Agony in Eight Fits by Lewis Carroll...
2,3,libro_78,Tarzan of the Apes,0.010302,0.010302,"...better preserved. Among other things he found a sharp hunting knife, on the keen blade of which he immediately proceeded to cut his finger. Undaunted he continued his experiments, finding th..."
3,4,libro_85,The Beasts of Tarzan,0.010284,0.010284,"...the old existence of constant bloodshed and danger—to the hunting and the being hunted. Grim beasts would stalk him, as they had stalked him in the past, and never would there be a moment, by sa..."
4,5,libro_66,The Dawn of Amateur Radio in the U.K. and Greece: A Personal View,0.009199,0.009199,"...was due to be made Artemis (Diana, the famous Goddess of Hunting) sent a small deer which was placed on the altar instead of the girl. Iphigenia was secretly spirited away to Taurida, in north..."



Consulta 2: 'internet network email computer users'
Necesidad: Guías o textos que expliquen qué es Internet y cómo usar el correo electrónico.
¿Puntuaciones iguales a sklearn (tolerancia 1e-09)?: True


,Posición,Identificador,Título,Puntuación,Puntuación sklearn,Fragmento
0,1,libro_75,E-mail 101,0.366332,0.366332,"...you 24 hour access to your bank account, you'll love the Internet. I suppose there are still people who, given a choice, will go to a drive-through teller just so they can deal with a ""live pe..."
1,2,libro_49,Surfing the Internet: An Introduction,0.352288,0.352288,"...Surfing the INTERNET: an Introduction Version 2.0.2 December 15, 1992 c. 1992 Jean Armour Polly. Material quoted from other authors was compiled from publ..."
2,3,libro_39,Hitchhiker's Guide to the Internet,0.336053,0.336053,...The Hitchhikers Guide to the Internet 25 August 1987 Ed Krol krol@uxc.cso.uiuc.edu...
3,4,libro_34,Zen and the Art of the Internet,0.299772,0.299772,...Part A Zen and the Art of the Internet Copyright (c) 1992 Brendan P. Kehoe Permission is granted to make and distribute verbatim copies of this guide provi...
4,5,libro_80,The Online World,0.247115,0.247115,...oice (registrations only): +47 370 31204 Internet mail: opresno@extern.uio.no Data/BBS: +47 370 31204 FAX: +47 370 27111...



Consulta 3: 'tarzan jungle apes'
Necesidad: Historias de Tarzán y sus aventuras en la selva.
¿Puntuaciones iguales a sklearn (tolerancia 1e-09)?: True


,Posición,Identificador,Título,Puntuación,Puntuación sklearn,Fragmento
0,1,libro_78,Tarzan of the Apes,0.482618,0.482618,...[Illustration] Tarzan of the Apes by Edgar Rice Burroughs Contents CHAPTER I. Out to Sea CHAPTER II. The Savage Home CHAPTER III. Life and Death CHAPTER I...
1,2,libro_81,The Return of Tarzan,0.477046,0.477046,...The Return Of Tarzan By Edgar Rice Burroughs Contents Chapter I The Affair on the Liner Chapter II Forging Bonds of Hate and ——? Chapter III What Happened in the...
2,3,libro_85,The Beasts of Tarzan,0.407754,0.407754,...[Illustration] The Beasts of Tarzan by Edgar Rice Burroughs To Joan Burroughs Contents CHAPTER I. Kidnapped CHAPTER II. Marooned CHAPTER III. Beasts at...
3,4,libro_92,Tarzan and the Jewels of Opar,0.333270,0.333270,...[Illustration] Tarzan and the Jewels of Opar by Edgar Rice Burroughs Contents CHAPTER I. Belgian and Arab CHAPTER II. On the Road To Opar CHAPTER III. The Ca...
4,5,libro_90,The Son of Tarzan,0.094096,0.094096,...[Illustration] The Son Of Tarzan by Edgar Rice Burroughs To Hulbert Burroughs Contents I. II. III. IV. V. VI. VII. VIII....



Consulta 4: 'constitution congress president rights government'
Necesidad: Documentos históricos sobre la Constitución y el gobierno de Estados Unidos.
¿Puntuaciones iguales a sklearn (tolerancia 1e-09)?: True


,Posición,Identificador,Título,Puntuación,Puntuación sklearn,Fragmento
0,1,libro_5,The United States Constitution,0.174577,0.174577,"...concerning possible corrections. *** THE CONSTITUTION OF THE UNITED STATES OF AMERICA, 1787 We the people of the United States, in Order to form a more perfect Union, e..."
1,2,libro_18,The Federalist Papers,0.146709,0.146709,...overnment FEDERALIST No. XIV. Objections to the Proposed Constitution From Extent of Territory Answered FEDERALIST No. XV. The Insufficiency of the Present Confederation to Preserve the Union...
2,3,libro_9,Abraham Lincoln's First Inaugural Address,0.117510,0.117510,"...and to take, in your presence, the oath prescribed by the Constitution of the United States, to be taken by the President “before he enters on the execution of his office.” I do not consider..."
3,4,libro_99,Collected Articles of Frederick Douglass,0.089022,0.089022,"...Mr. Johnson told me that there was nothing in the laws or constitution of Massachusetts that would prevent a colored man from being governor of the State, if the people should see fit to elect him..."
4,5,libro_48,The 1992 CIA World Factbook,0.077549,0.077549,...(Nuristan) Independence: 19 August 1919 (from UK) Constitution: the old Communist-era constitution probably will be replaced with an Islamic constitution Legal system: a...
